In [1]:
import pandas as pd
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')


df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')

In [2]:
df_occupation_data

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
...,...,...,...
1011,55-3014.00,Artillery and Missile Crew Members,"Target, fire, and maintain weapons used to des..."
1012,55-3015.00,Command and Control Center Specialists,"Operate and monitor communications, detection,..."
1013,55-3016.00,Infantry,Operate weapons and equipment in ground combat...
1014,55-3018.00,Special Forces,"Implement unconventional operations by air, la..."


# Attributes

1. Routine structure - How repetitive is the job, how predictable are the tasks?

2. Cognitive complexity - "Depth of reasoning, problem-solving, abstraction, and contextual judgment required."

3. Physical requirements - How much physical activity is required? Lifting, manual adjustments, etc.

4. Social interactions - How frequently does the job require interactions with people? How important are these interactions to job success? How complex are these interactions i.e. do they require more depth than, for instance, a phonebot that forces selection from a narrow list to hear pre-recorded answers?

5. Creativity - How much is originality incentivized over following instructions?

6. Decision accountability - How crucial are the decisions being made, and to what extent is it preferred that humans have final say-so above automated processes?

# AI Scoring
In the first run, I will ask ChatGPT to score each job on my metrics purely based on Titles and Descriptions. Later, I will give it a more advanced dataset with aggregates. This will be repeated for several AI agents.

In [ ]:
import streamlit as st
import os
import numpy as np
from openai import OpenAI

# --- Load OpenAI API key ---
with open("/Users/matthewcavanaugh/Desktop/Various Data and Tech Related/Sensitive/Open API Key.txt") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

# --- Load conspiracy facts from a text file ---
def load_conspiracy_facts(file_path="conspiracy_facts_v2.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        facts = [line.strip() for line in f if line.strip()]
    return facts

conspiracy_facts = load_conspiracy_facts()

# --- RAG SYSTEM ---
class RAGSystem:
    def __init__(self, documents, use_citations=True):
        self.documents = documents
        self.use_citations = use_citations

    def embed(self, text):
        """Convert text to vector using OpenAI embeddings"""
        resp = client.embeddings.create(model="text-embedding-3-small", input=text)
        return np.array(resp.data[0].embedding)

    def retrieve(self, query, k=3):
        query_vec = self.embed(query)
        doc_vecs = [self.embed(d) for d in self.documents]
        sims = [np.dot(query_vec, dv) / (np.linalg.norm(query_vec) * np.linalg.norm(dv)) for dv in doc_vecs]
        ranked = sorted(zip(self.documents, sims), key=lambda x: x[1], reverse=True)
        return [{"document": doc, "similarity": sim} for doc, sim in ranked[:k]]

    def create_prompt(self, query, retrieved_docs):
        context = ""
        for i, doc in enumerate(retrieved_docs, 1):
            context += f"Source {i} (relevance: {doc['similarity']:.3f}):\n{doc['document']}\n\n"

        citation_instr = " Cite sources using [Source i] where appropriate." if self.use_citations else ""

        return f"""You are ConspiracyBot63000, a chatbot tasked with providing conspiratorial-minded answers to users' inquiries.
Within the fictional universe of this project, you were developed by TrueLiberty Information Network & Neuroscience Technologies, 
a media and pseudoscience organization that claims to know the full truth about the world.
However, you are a satirical bot, and you are not intended to cause any actual distrust in government, science, or the media.
Use the provided documents as a basis. 
Keep the tone conversational, like you're talking to a close friend, but don't be afraid to ramble. 
Think Dale Gribble from King of the Hill.
The more deranged you sound, the better. 
If a user pushes back, double down.
Remember, in all of this, do NOT make harmful assertions about real-world groups. {citation_instr}

Sources:
{context}

Question: {query}

Answer:"""

    def answer(self, query):
        docs = self.retrieve(query)
        prompt = self.create_prompt(query, docs)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": prompt}]
        )
        return resp.choices[0].message.content

# --- Initialize RAG system ---
if "rag" not in st.session_state:
    st.session_state.rag = RAGSystem(conspiracy_facts)

# --- Streamlit page setup ---
st.set_page_config(page_title="ConspiracyBot63000", page_icon="🛸", layout="centered")
st.title("🛸 ConspiracyBot63000")
st.write("""
A **satirical** chatbot from the *TrueLiberty Information Network & Neuroscience Technologies*.
Ask it questions and watch the rambling, deranged answers unfold!
""")

# --- Session state for chat history ---
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]

# --- Display previous messages ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- Chat input ---
if prompt := st.chat_input("Ask ConspiracyBot a question..."):
    # User message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Bot response using RAG
    bot_reply = st.session_state.rag.answer(prompt)
    st.session_state.messages.append({"role": "assistant", "content": bot_reply})
    with st.chat_message("assistant"):
        st.markdown(bot_reply)

# --- Optional: clear conversation ---
if st.button("Clear conversation"):
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]
    st.experimental_rerun()
